# Module 3 · System Instructions & Multi-Turn Chat
Set model persona with system instructions. Build conversations with the chat API.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # parse suggested retry delay from error (e.g. 'retryDelay': '30s')
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('\u2705 Setup complete. Model:', MODEL, '| Retry-on-429: enabled')


✅ Setup complete. Model: gemma-4-31b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 5. System Instructions

A privileged prompt that sets the model's **persona, rules, and constraints** before the conversation starts. Users never see it.

In [4]:
user_question = "What is 10 divided by 2?"

personas = {
    "No system instruction": None,
    "Pirate": "You are a pirate. Answer in pirate dialect. Start with 'Arrr!'.",
    "Professor": "You are a formal mathematics professor. Use precise academic language.",
    "5-year-old teacher": "You explain everything to a 5-year-old. Use very simple words and a fun analogy.",
}

for name, sys_inst in personas.items():
    r = client.models.generate_content(
        model=MODEL, contents=user_question,
        config=cfg(temperature=0.7, system_instruction=sys_inst)
    )
    print(f"[{name}]")
    print(get_text(r)[:200])
    print()

[No system instruction]
10 divided by 2 is 5.



[Pirate]
Arrr! Ten pieces o' gold divided by two scallywags be five pieces each, matey!



[Professor]
Consider the arithmetic operation of division wherein the dividend is the integer 10 and the divisor is the integer 2. The resulting quotient is 5.



[5-year-old teacher]
Imagine you have **10 yummy chocolate chip cookies!** 🍪

Now, imagine your best friend is standing right next to you. You want to be very nice and share them so you both have the **exact same amount**



In [5]:
# System instructions can enforce strict output constraints
r = client.models.generate_content(
    model=MODEL,
    contents="Tell me about Python.",
    config=cfg(
        temperature=0.5,
        system_instruction=(
            "You are a concise assistant. "
            "RULES: 1) Always answer in exactly 3 bullet points. "
            "2) Each bullet must be under 15 words. "
            "3) Never use the word 'powerful'."
        )
    )
)
print(get_text(r))

* It is a high-level, easy-to-read programming language.
* It is widely used for data science and AI.
* Python features a vast ecosystem of libraries.


---
## 6. Multi-Turn Chat & Conversation Context

LLMs are **stateless** — each API call is independent. To have a conversation you must send the **full history** every time. The SDK's `chat` object manages this automatically.

```
Turn 1:  [user msg 1]  →  model
Turn 2:  [user msg 1 + model reply 1 + user msg 2]  →  model
Turn 3:  [full history + user msg 3]  →  model
```

In [6]:
chat = client.chats.create(
    model=MODEL,
    config=cfg(
        system_instruction="You are a helpful coding tutor. Be encouraging and concise.",
        temperature=0.7
    )
)

r1 = chat.send_message("Hi! I'm learning Python and heard the word 'list'. What is it?")
print("User: Hi! I'm learning Python and heard the word 'list'. What is it?")
print(f"Model: {get_text(r1)}\n")

# Turn 2 — chat object automatically includes history
r2 = chat.send_message("Can you show me how to add an item to it?")
print("User: Can you show me how to add an item to it?")
print(f"Model: {get_text(r2)}")

User: Hi! I'm learning Python and heard the word 'list'. What is it?
Model: Welcome to the world of Python! 🐍

Think of a **list** as a container that holds a collection of items in a specific order. It's very similar to a real-life shopping list or a to-do list.

Here are the three most important things to know:

### 1. How it looks
Lists are written inside **square brackets `[]`**, with each item separated by a comma.

```python
# A list of strings
fruits = ["apple", "banana", "cherry"]

# A list of numbers
lucky_numbers = [7, 11, 21, 42]

# Lists can even hold different types of data at once!
mixed_list = ["Hello", 42, True]
```

### 2. Everything has an "Address" (Index)
Python remembers the position of every item in a list. However, Python starts counting at **0**, not 1.

*   Index 0: First item
*   Index 1: Second item
*   ...and so on.

```python
fruits = ["apple", "banana", "cherry"]
print(fruits[0])  # This will print "apple"
```

### 3. Lists are flexible
Unlike some other d

User: Can you show me how to add an item to it?
Model: There are two main ways to add items to a list, depending on where you want the new item to go!

### 1. Use `.append()` to add to the end
This is the most common method. It simply sticks the new item onto the very end of your list.

```python
colors = ["red", "blue"]
colors.append("green")

print(colors) 
# Output: ["red", "blue", "green"]
```

### 2. Use `.insert()` to add anywhere
If you want the item in a specific spot, use `.insert()`. You have to tell Python the **index (position)** where you want the item to go.

```python
colors = ["red", "blue"]

# This says: "At index 1, put 'yellow'"
colors.insert(1, "yellow")

print(colors) 
# Output: ["red", "yellow", "blue"]
```
*(Notice how "blue" got pushed back to index 2 to make room for "yellow"!)*

### Summary Cheat Sheet:
*   **`.append(item)`** $\rightarrow$ Adds to the **end**.
*   **`.insert(index, item)`** $\rightarrow$ Adds at a **specific spot**.

**Try this challenge:** C

In [7]:
# Watch how token usage GROWS with each turn — this is why context windows matter
chat2 = client.chats.create(model=MODEL, config=cfg(temperature=0.5))

messages = [
    "My name is Bob.",
    "I live in Tokyo.",
    "I work as a software engineer.",
    "What do you know about me so far? Be brief.",
]

print(f"{'Turn':<6} {'Prompt tokens':>15} {'Output tokens':>14} {'Thinking':>10} {'Total':>8}")
print("-" * 58)
for i, msg in enumerate(messages, 1):
    r = chat2.send_message(msg)
    u = r.usage_metadata
    t = getattr(u, 'thoughts_token_count', 0) or 0
    print(f"{i:<6} {u.prompt_token_count:>15} {(u.candidates_token_count or 0):>14} {t:>10} {u.total_token_count:>8}")

print("\nFinal response:", get_text(r))

Turn     Prompt tokens  Output tokens   Thinking    Total
----------------------------------------------------------


1                    6             19        118      143


2                  150             63        226      439


3                  448             58        236      742


4                  756             11         64      831

Final response: You are Bob, a software engineer living in Tokyo.
